---
**Navigation** : [<< Sudoku-12 Z3 C#](Sudoku-12-Z3-Csharp.ipynb) | [Index](README.md) | [Sudoku-14 Retour d'experience >>](Sudoku-14-BDD-Csharp.ipynb)

---

# Sudoku-13 : Le Sudoku comme Regex Symbolique - l'echelle Conway -> BREX/Rex -> RE#

> **Ce notebook reprend et corrige une tentative personnelle de 2020** : representer un Sudoku comme un *grand regex symbolique* ou les contraintes de lignes, colonnes et blocs se combinent par **intersection** (`Ligne & Colonne & Bloc`), puis demander a un solveur d'en extraire une grille-temoin.
>
> La tentative de 2020 a bute sur deux murs reels - tous deux des murs **de l'automate** (determinisation explosive + temoin cape). La chaine moderne change de moteur : les operateurs de surface `&` / `~` se compilent vers la **theorie des chaines de Z3** (`re.inter` / `re.comp`), qui ne construit aucun produit d'automates et n'a aucun plafond de temoin. Les deux murs de 2020 disparaissent donc. Mais resoudre ne devient pas gratuit : la difficulte **migre** dans le solveur de chaines de Z3 - instantanee pour un `A & ~B` general, lente pour une ligne Sudoku, et hors de portee pour la grille 81. Le Sudoku, lui, se resout par le bon encodage : `Distinct` sur 81 entiers.

## La these en une phrase

Un Sudoku se laisse decrire comme une **intersection de contraintes regulieres**. Mais *decrire* (reconnaitre une grille valide) et *produire* (resoudre le puzzle) sont **deux metiers differents**. Changer de moteur (automate -> theorie des chaines Z3) fait tomber les murs de 2020, mais ne supprime pas la difficulte : elle se deplace, et le bon outil depend de la **forme** du probleme - `Distinct` entier pour le Sudoku, generation de temoin `A & ~B` pour les cas generaux, RE# pour la reconnaissance.

## 1. Introduction : un Sudoku comme grand regex ?

En 2020, l'auteur entreprenait de representer un Sudoku non pas comme un monstre PCRE a backtracking (le folklore Perl du "Sudoku en un regex", barreau 1 ci-dessous), mais comme une **chaine declarative tractable** ou :

- chaque contrainte (ligne, colonne, bloc) est un *regex symbolique* ;
- les contraintes se combinent par **intersection** (`&`) et **complement** (`~`) ;
- l'intersection compile vers un terme SMT ;
- un solveur SMT (Z3) en extrait un **modele = la grille solution** (un *temoin*).

Ce notebook raconte **honnetement** cette histoire : ses deux murs de 2020, la chaine moderne qui les fait tomber en changeant de moteur, et la lecon plus fine que "un mur tombe" - la difficulte ne disparait pas, elle se **deplace**.

### Reconnaitre != resoudre

La distinction centrale de cette serie, rendue vivante par le Sudoku :

| Approche | **RESOUDRE** (produire le temoin) | **VERIFIER** (valider une grille) |
|---|---|---|
| Folklore PCRE (backtracking, barreau 1) | detour de backtracking, illisible | monstrueux |
| 2020 - AutomataDotNet (BREX/Rex + SFAz3) | **mure** : cap temoin ~21 char (#6) | intersection -> DFA, mais **explose** |
| Moderne - `&`/`~` -> theorie des chaines Z3 | **temoin non cape, sans produit d'automates** (rapide pour `A & ~B`, lent pour une ligne) | (oui) |
| 2025 - RE# | aucun temoin (recognition-only) | **temps lineaire** |
| Z3 `Distinct` (81 entiers) | **resolveur de production du Sudoku** (~47 ms) | - |

Les deux murs de 2020 etaient des murs **de l'automate** : ils tombent ensemble des qu'on change de moteur (DFA -> theorie des chaines Z3). Mais aucun outil ne "couronne" l'echelle : RE# n'apporte que la moitie *verification* (temps lineaire), Z3 string-theory genere le temoin general mais peine sur le Sudoku, et c'est `Distinct` sur des entiers - pas du tout un regex - qui resout reellement la grille. Le bon outil suit la **forme** du probleme.

## 2. Barreau 1 - le monstre PCRE : "le Sudoku en un seul regex"

Le folklore Perl du "Sudoku en un seul regex" remonte aux PerlMonks du milieu des annees 2000 ([ikegami, 2005](https://www.perlmonks.org/?node_id=471168) ; plus tard le module pur-regex [`Regexp::Sudoku` d'Abigail](https://metacpan.org/pod/Regexp::Sudoku)). L'idee : le backtracking du moteur regex *est* un moteur de recherche, et la grille devient un unique match potentiel.

Ce tour de force existe en **deux formes reelles** (souvent confondues) :

- la **forme recursive** (`(?R)` / `(?0)`) : le motif s'auto-invoque pour explorer la grille. Elle est **non reguliere** et **hors de portee de .NET** (`System.Text.RegularExpressions` ne fait pas de recursion de motif) ;
- la **forme deroulee** : les 81 cases sont ecrites explicitement (backreferences + lookaheads, **sans** `(?R)`), lignee Aron Griffis (2007), domaine public. Elle **tourne** sur tout moteur a backtracking, .NET compris.

Dans les deux cas, le pattern est un **monstre** illisible, inversement pedagogique, et non generalisable - du folklore de concours, pas une methode. La forme deroulee est celle que nous **generons et executons reellement** en section 8 ; le tronceau ci-dessous en est extrait (un troncage de l'artefact reel sauvegarde dans `assets/sudoku-unrolled.regex.txt`), pas un fragment reconstruit a la main.

In [1]:
// Le VRAI monstre regex (forme deroulee, lignee Conway/folklore PCRE) genere en section 8
// et sauvegarde dans assets/sudoku-unrolled.regex.txt. On en affiche un TRONCAGE : tete + queue,
// extraits du fichier reel (pas un fragment reconstruit a la main).
using System;
using System.IO;
using System.Linq;

string[] candidats = {
    "assets/sudoku-unrolled.regex.txt",
    "MyIA.AI.Notebooks/Sudoku/assets/sudoku-unrolled.regex.txt",
    Path.Combine("..", "Sudoku", "assets", "sudoku-unrolled.regex.txt")
};
string chemin = candidats.FirstOrDefault(File.Exists);

if (chemin == null) {
    Console.WriteLine("(artefact assets/sudoku-unrolled.regex.txt absent - il est (re)genere en section 8)");
} else {
    string monstre = File.ReadAllText(chemin);
    int n = monstre.Length;
    string tete  = monstre.Substring(0, Math.Min(420, n));
    string queue = n > 720 ? monstre.Substring(n - 300) : "";
    Console.WriteLine($"Monstre regex deroule (forme exacte, {n} caracteres) - TRONCAGE de {Path.GetFileName(chemin)} :");
    Console.WriteLine("--- tete (420 premiers caracteres) -------------------------------");
    Console.WriteLine(tete);
    Console.WriteLine($"--- [... {n - 720} caracteres omis ...] --------------------------");
    Console.WriteLine(queue);
    Console.WriteLine("------------------------------------------------------------------");
    Console.WriteLine();
    Console.WriteLine("-> Backtracking PCRE detourne en moteur de recherche : ca 'tourne' (section 8),");
    Console.WriteLine("   mais l'unicite encodee en lookaheads/backrefs est illisible. Ce n'est PAS");
    Console.WriteLine("   la representation declarative que nous cherchons (intersection -> Z3).");
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

Monstre regex deroule (forme exacte, 13515 caracteres) - TRONCAGE de sudoku-unrolled.regex.txt :


--- tete (420 premiers caracteres) -------------------------------


\A

\d*(\d)
(?!(?:.*\n)+(?:.{10}){0}\1\b)
(?!\d*\ (?:.{10})*?\1\b)
(?!\d*\ (?:.{10}){0,1}\1\b)
(?!(?:.*\n){1,2}(?:.{30}){0}(?:.{10}){0,2}\1\b)
\d*\s+

\d*(?!\1)
(\d)
(?!(?:.*\n)+(?:.{10}){1}\2\b)
(?!\d*\ (?:.{10})*?\2\b)
(?!\d*\ (?:.{10}){0,0}\2\b)
(?!(?:.*\n){1,2}(?:.{30}){0}(?:.{10}){0,2}\2\b)
\d*\s+

\d*(?!\1|\2)
(\d)
(?!(?:.*\n)+(?:.{10}){2}\3\b)
(?!\d*\ (?:.{10})*?\3\b)
(?!(?:.*\n){1,2}(?:.{30}){0}(?:.{10}){0,2}


--- [... 12795 caracteres omis ...] --------------------------


|\70|\71|\72|\73|\74|\75|\76|\77|\78|\79)
(\d)
(?!(?:.*\n)+(?:.{10}){7}\80\b)
(?!\d*\ (?:.{10})*?\80\b)
(?!\d*\ (?:.{10}){0,0}\80\b)
\d*\s+

\d*(?!\9|\18|\27|\36|\45|\54|\61|\62|\63|\70|\71|\72|\73|\74|\75|\76|\77|\78|\79|\80)
(\d)
(?!(?:.*\n)+(?:.{10}){8}\81\b)
(?!\d*\ (?:.{10})*?\81\b)
\d*\s+

\Z



------------------------------------------------------------------


-> Backtracking PCRE detourne en moteur de recherche : ca 'tourne' (section 8),


   mais l'unicite encodee en lookaheads/backrefs est illisible. Ce n'est PAS


   la representation declarative que nous cherchons (intersection -> Z3).


### Interpretation : le monstre PCRE = l'anti-modele

Ce monstre demontre par l'absurde que **detourner le backtracking d'un moteur regex pour faire de la recherche** conduit a une representation illisible. La contrainte d'unicite Sudoku n'est *pas* naturelle en PCRE : elle s'exprime en empilant des lookaheads negatifs sur les cellules deja posees. La forme deroulee *tourne* (section 8) mais reste illisible et, on le verra, non portable d'un moteur a l'autre ; la forme recursive `(?R)` ne tourne meme pas en .NET.

La bonne representation, c'est l'**intersection declarative** de contraintes (`Ligne & Colonne & Bloc`) - portee proprement par RE# pour la reconnaissance (sections 4-5) et par Z3 pour la production du temoin (section 6). C'est exactement ce que cherchait l'approche 2020.

## 3. Barreau 2 - La tentative 2020 : BREX, Rex et le mur

En 2020, l'auteur s'appuie sur la chaine d'outils symboliques de Margus Veanes (**AutomataDotNet**) pour realiser l'intersection declarative. Deux briques existaient, verifiees par lecture du source au commit `0242132f` :

**1. L'intersection existait - via BREX** (Boolean combinations of Regular EXpressions, fichiers `BREX.cs`, `BREXManager.cs`). Mais c'etait une **API builder C#**, pas un operateur `&` de surface dans une chaine :

```csharp
var man  = new BREXManager();
var like1 = man.MkLike(@"%[ab]_____");
var like2 = man.MkLike(@"%[bc]_____");
var and   = man.MkAnd(like1, like2);   // l'intersection : une METHODE, pas un '&' dans la chaine
var dfa   = and.Optimize();
```

**2. Le temoin Z3 existait - via la lignee Rex** (`RegexToSMTConverter.cs`) : generation d'un membre/temoin d'un regex par SMT. C'est ce chemin qui a bute sur la troncature a 21 caracteres.

### Les deux murs reels (verifies dans les tests)

| Mur | Preuve dans le source | Consequence |
|-----|----------------------|-------------|
| **Explosion d'etats** | `BREXTests.cs` porte les commentaires `//fails due to timeout` et `//fails due to too many states` - l'intersection de deux `MkLike` de **8-9 underscores seulement** fait exploser le DFA via `Optimize()` | L'intersection symbolique est trop chere a determiniser |
| **Cap du temoin a 21 caracteres** | Issue upstream [AutomataDotNet/Automata#6](https://github.com/AutomataDotNet/Automata/issues/6) (creee le 2020-12-07, **toujours 0 reponse**) : le chemin SFAz3+Z3 tronque le modele | On ne peut pas produire une grille-temoin完整e |

Ci-dessous, le builder BREX reproduit en **pseudo-code documentaire** (non execute - AutomataDotNet est un projet Microsoft abandonne, non restaure ici). Le but est de montrer la *forme* de l'approche 2020 : un spaghetti de C# et de strings, pas encore la chaine monolithique elegante.

In [2]:
// Pseudo-code DOCUMENTAIRE de l'approche 2020 (BREX + Rex + Z3), non execute.
// Montre la FORME : un builder C# d'intersections, pas un operateur de surface.
//
// var man = new BREXManager();
// // une ligne Sudoku valide = intersection de contraintes
// var lineConstraint = man.MkAnd(
//     man.MkLike("9 chiffres 1-9"),
//     man.MkAnd(man.MkLike("contient 1"), man.MkAnd(man.MkLike("contient 2"), ...)));
// var fullSudoku = man.MkAnd(man.MkAnd(lignes), man.MkAnd(man.MkAnd(colonnes), man.MkAnd(blocs)));
// var dfa = fullSudoku.Optimize();   // <-- MUR 1 : explosion d'etats sur 8-9 "underscores"
// var witness = RexToSMT(dfa);        // <-- MUR 2 : cap 21 caracteres (issue #6)

Console.WriteLine("Approche 2020 (documentaire, non execute) :");
Console.WriteLine("  - Intersection via BREX builder C# (MkAnd/MkLike), pas un '&' de surface");
Console.WriteLine("  - Mur 1 : Optimize() explose le DFA (BREXTests.cs '//too many states')");
Console.WriteLine("  - Mur 2 : temoin Z3 tronque a 21 char (AutomataDotNet/Automata#6, 0 reponse depuis 2020-12-07)");
Console.WriteLine();
Console.WriteLine("Conclusion : l'intuition declarative (intersection) etait juste et");
Console.WriteLine("contemporaine du papier Veanes (MSR-TR-2020-25, aout 2020).");
Console.WriteLine("Mais la FORME etait un spaghetti C#, et les deux murs etaient reels.");

Approche 2020 (documentaire, non execute) :


  - Intersection via BREX builder C# (MkAnd/MkLike), pas un '&' de surface


  - Mur 1 : Optimize() explose le DFA (BREXTests.cs '//too many states')


  - Mur 2 : temoin Z3 tronque a 21 char (AutomataDotNet/Automata#6, 0 reponse depuis 2020-12-07)


Conclusion : l'intuition declarative (intersection) etait juste et


contemporaine du papier Veanes (MSR-TR-2020-25, aout 2020).


Mais la FORME etait un spaghetti C#, et les deux murs etaient reels.


### Interpretation : deux murs, tous deux des murs de l'automate

L'approche 2020 n'etait pas le folklore PCRE : l'intuition declarative par intersection etait **fondue et contemporaine** des travaux de Veanes. Mais la **forme** (builder C# `MkAnd(MkLike(...), MkLike(...))`) etait un spaghetti, et **deux murs reels** l'ont bloquee :

1. l'intersection symbolique **explose** a la determinisation (produit de DFA) ;
2. le chemin temoin **SFAz3 -> Z3 est cape** a ~21 caracteres ([#6](https://github.com/AutomataDotNet/Automata/issues/6)).

Ces deux murs ont la **meme racine** : on passe par la *materialisation d'un automate* (determinisation, puis extraction de temoin par ce chemin). Deux reparations, deux registres :

- pour la **reconnaissance**, RE# (section 4) rend l'intersection lineaire sans determinisation exponentielle ;
- pour la **production de temoin**, la chaine moderne (section 6b) compile `&`/`~` vers la **theorie des chaines de Z3** : plus de produit d'automates (donc plus le mur 1) et plus de cap (mur 2 leve). Mais la difficulte ne s'evapore pas - elle migre dans le solveur de chaines, comme on le mesurera.

## 4. Barreau 3 - RE# (2025) : la reconnaissance enfin tractable

[RE#](https://github.com/ieviev/resharp-dotnet) (NuGet `Resharp`, engine F#/.NET, MIT) etend la syntaxe `System.Text.RegularExpressions` avec **trois operateurs first-class** :

- `&` : **intersection** (`A & B` reconnait les mots acceptes par A ET par B) ;
- `~` : **complement** (`~A` reconnait tout ce que A ne reconnait pas) ;
- `_` : wildcard universel.

Crucialement, RE# est **non-backtracking** et compile vers des automates dont la reconnaissance est **temps lineaire** - grace a la *finitude des derivees symboliques* (formalisee en Lean, cf section references). L'intersection de deux RE# ne determinise **pas** de maniere exponentielle : c'est precisement le mur 1 de 2020 qui tombe.

**Caveat honnete** : RE# est **recognition-only**. Il *reconnait* (valide) ; il ne *genere* aucun temoin. Pour produire une grille, il faudra Z3 (section 6).

> **Note technique (environnement)** : `Resharp` est un **package NuGet standard** (non modifie), mais `#r "nuget: Resharp"` ne suffit pas sous le kernel `.net-csharp` : le restore aboutit, puis RE# echoue **a l'execution** avec `FileNotFoundException: FSharp.Core, Version=10.0.0.0` - le kernel ne lie pas le runtime F# via une reference NuGet. On charge donc les DLL **par chemin relatif** vers un dossier `.deploy` **committe dans le depot** (`SymbolicAI/SMT/Resharp/.deploy/` : `Resharp.dll`, `Resharp.Runtime.dll` et surtout `FSharp.Core.dll` 10.x, assembly 10.0.0.0). Resultat : reproductible apres un simple clone, hors-ligne, sans aucun chemin utilisateur code en dur.

In [3]:
// RE# via DLL in-repo (.deploy, portable) : pas de #r "nuget:" sous papermill, et plus aucun chemin utilisateur code en dur.
// Resharp + Resharp.Runtime (net8.0) + FSharp.Core 10.x (assembly 10.0.0.0) co-localises dans le dossier .deploy committe ;
// chemin relatif au notebook, reproductible sur toute machine. Le kernel .net-csharp tourne sous net8.0.
#r "../SymbolicAI/SMT/Resharp/.deploy/FSharp.Core.dll"
#r "../SymbolicAI/SMT/Resharp/.deploy/Resharp.Runtime.dll"
#r "../SymbolicAI/SMT/Resharp/.deploy/Resharp.dll"
using System.Diagnostics;

// Smoke test : l'INTERSECTION & est first-class dans UNE chaine. C'est l'operateur
// qui encode la contrainte de ligne Sudoku (section suivante). On le demontre ici
// en combinant 2 puis 3 contraintes d'inclusion dans la meme expression.
var sw = Stopwatch.StartNew();
var has5And7 = new Resharp.Regex(@".*5.*&.*7.*");          // contient un 5 ET un 7
var has123   = new Resharp.Regex(@".*1.*&.*2.*&.*3.*");    // contient 1, 2 ET 3 (3-way)
sw.Stop();
Console.WriteLine($"RE# compile en {sw.ElapsedMilliseconds} ms (premier appel, JIT inclus).");
Console.WriteLine($"'527' contient 5 et 7     : {(has5And7.Matches("527").Length > 0 ? "oui" : "non")}");
Console.WriteLine($"'521' contient 5 et 7     : {(has5And7.Matches("521").Length > 0 ? "oui" : "non")}");
Console.WriteLine($"'123456789' a 1, 2 et 3   : {(has123.Matches("123456789").Length > 0 ? "oui" : "non")}");
Console.WriteLine($"'456789' a 1, 2 et 3      : {(has123.Matches("456789").Length > 0 ? "oui" : "non")}");
Console.WriteLine();
Console.WriteLine("RE# supporte aussi le complement ~ et le wildcard _ (first-class, documentes");
Console.WriteLine("par le moteur). Nous nous appuyons ici sur l'INTERSECTION &: c'est elle qui");
Console.WriteLine("encode de maniere robuste la contrainte de ligne Sudoku ci-dessous.");


RE# compile en 207 ms (premier appel, JIT inclus).


'527' contient 5 et 7     : oui


'521' contient 5 et 7     : non


'123456789' a 1, 2 et 3   : oui


'456789' a 1, 2 et 3      : non


RE# supporte aussi le complement ~ et le wildcard _ (first-class, documentes


par le moteur). Nous nous appuyons ici sur l'INTERSECTION &: c'est elle qui


encode de maniere robuste la contrainte de ligne Sudoku ci-dessous.


### Interpretation : RE# valide, ne resout pas

RE# rend l'**intersection** (`&`) et le **complement** (`~`) first-class dans une chaine unique, compilee en temps lineaire. C'est precisement le barreau intermediaire qui manquait en 2020 : fini le spaghetti builder `MkAnd(MkLike(...))`, fini l'explosion DFA.

Mais notons le caveat deja mentionne : RE# repond `matche` ou `ne matche pas`. Il ne dit pas *quelle* chaine satisferait le pattern. C'est un **verificateur**, pas un resolveur.

## 5. RE# verificateur d'une ligne Sudoku remplie

Encodons la **contrainte de ligne** d'un Sudoku comme une intersection RE# :

> Une ligne valide = exactement 9 chiffres 1-9 (`[1-9]{9}`) ET elle contient un 1 ET un 2 ... ET un 9.

Soit, litteralement :

```
[1-9]{9} & .*1.* & .*2.* & .*3.* & .*4.* & .*5.* & .*6.* & .*7.* & .*8.* & .*9.*
```

Cette intersection de 10 regex est **exactement** le type d'operation qui faisait exploser le DFA en 2020 (`BREXTests.cs` : "//too many states" sur 8-9 underscores). Avec RE#, elle compile et s'execute en temps lineaire.

In [4]:
using System.Diagnostics;
// La contrainte de ligne Sudoku comme intersection RE# (10 sous-regex).
// En 2020 cette intersection explosait le DFA ; RE# la compile en temps lineaire.
var sw = Stopwatch.StartNew();
var ligneValide = new Resharp.Regex(
    @"[1-9]{9}&.*1.*&.*2.*&.*3.*&.*4.*&.*5.*&.*6.*&.*7.*&.*8.*&.*9.*");
sw.Stop();
Console.WriteLine($"Contrainte de ligne (10-way intersection) compilee en {sw.ElapsedMilliseconds} ms.");

void VerifierLigne(string ligne, string attendu)
{
    bool ok = ligneValide.Matches(ligne).Length > 0;
    string verdict = ok ? "VALIDE" : "invalide";
    Console.WriteLine($"  '{ligne}' -> {verdict}  (attendu : {attendu}) {(ok == (attendu == "VALIDE") ? "[OK]" : "[ECHEC]")}");
}

Console.WriteLine("\nLignes valides (permutations de 1..9) :");
VerifierLigne("123456789", "VALIDE");
VerifierLigne("987654321", "VALIDE");
VerifierLigne("534678912", "VALIDE");
Console.WriteLine("\nLignes invalides :");
VerifierLigne("123456788", "invalide");   // doublon (8 deux fois)
VerifierLigne("12345678",  "invalide");   // trop court
VerifierLigne("112345678", "invalide");   // doublon (1 deux fois)
VerifierLigne("123456780", "invalide");   // 0 interdit

Contrainte de ligne (10-way intersection) compilee en 87 ms.



Lignes valides (permutations de 1..9) :


  '123456789' -> VALIDE  (attendu : VALIDE) [OK]


  '987654321' -> VALIDE  (attendu : VALIDE) [OK]


  '534678912' -> VALIDE  (attendu : VALIDE) [OK]



Lignes invalides :


  '123456788' -> invalide  (attendu : invalide) [OK]


  '12345678' -> invalide  (attendu : invalide) [OK]


  '112345678' -> invalide  (attendu : invalide) [OK]


  '123456780' -> invalide  (attendu : invalide) [OK]


### Interpretation : RE# verifie en O(n), lineaire

La contrainte de ligne - une intersection de 10 regex - compile et s'execute en temps lineaire. **C'est le mur 1 de 2020 qui tombe** : la meme operation qui faisait exploser le DFA (`//too many states`) est desormais tractable.

Ce verificateur reconnait une ligne valide. Applique a chacune des 9 lignes d'une grille remplie, il valide la grille entiere (apres extraction des colonnes et blocs, memes contraintes). **Mais il ne produit toujours aucune grille** : donnez-lui un puzzle vide, il ne saura pas le remplir.

## Exercice 1 : une contrainte d'intersection RE#

# Objectif
Ecrivez un pattern RE# qui reconnait une ligne Sudoku valide **ET** dans laquelle le chiffre **5 apparaît avant le 7** (sous-sequence `5 ... 7`).

# Indice
Composez par intersection la contrainte de ligne valide (ci-dessus) avec `.*5.*7.*` (un 5 puis, plus loin, un 7). RE# accepte l'imbrication libre de `&` dans une meme chaine.

# Etape 1
Definissez le pattern combine dans la cellule suivante. Une permutation comme `123456789` (5 avant 7) doit passer ; `987654321` (7 avant 5) doit echouer.

In [5]:
// EXERCICE : ligne Sudoku valide AVEC 5 avant 7 (intersection & ordre)
// TODO etudiant : completer le pattern ci-dessous.
// Indice : croisez la contrainte de ligne valide avec .*5.*7.* (5 avant 7).
string ligneAvec5Avant7 = "[1-9]{9}";  // TODO etudiant : completer l'intersection
var exoRe = new Resharp.Regex(ligneAvec5Avant7);
Console.WriteLine("Exercice a completer : 5 doit apparaitre avant 7.");
Console.WriteLine($"  '123456789' (5 avant 7) -> valide attendu, obtenu : {(exoRe.Matches("123456789").Length > 0 ? "valide" : "rejet")}");
Console.WriteLine($"  '987654321' (7 avant 5) -> rejet attendu,  obtenu : {(exoRe.Matches("987654321").Length > 0 ? "valide" : "rejet")}");

Exercice a completer : 5 doit apparaitre avant 7.


  '123456789' (5 avant 7) -> valide attendu, obtenu : valide


  '987654321' (7 avant 5) -> rejet attendu,  obtenu : valide


## 6. Le resolveur - Z3 produit le temoin

RE# sait *verifier* ; il ne sait pas *produire*. Pour resoudre un puzzle Sudoku, il faut un **resolveur**. C'est le role de Z3, et c'est le barreau 2 de 2020 - mais cette fois **sans le cap de 21 caracteres**, parce qu'on appelle Z3 **directement** (sans passer par le chemin SFAz3 de AutomataDotNet qui etait mure).

L'idee : encoder les contraintes `Ligne & Colonne & Bloc` non plus comme un regex symbolique, mais comme des **assertions SMT** sur 81 variables entieres, puis demander a Z3 un **modele = la grille solution**.

> **Note technique** : comme pour RE#, on charge Z3 via reference DLL a **chemin relatif** + un `NativeLibrary.SetDllImportResolver` pour la librairie native `libz3.dll` (le `#r "nuget:"` se bloquerait sous papermill). Les binaires Z3 et le fork Automata (section 6b) vivent dans le `.deploy` de leurs **submodules** respectifs (`SymbolicAI/SMT/Z3.Linq` et `.../Automata`, serie Z3) - a initialiser via `git submodule update --init`. Plus aucun chemin utilisateur code en dur.

In [6]:
// Z3 chargee par chemin RELATIF au depot (.deploy) + resolver natif (libz3.dll), pas de #r "nuget:" sous papermill.
// Plus aucun chemin utilisateur code en dur. Les binaires Z3 proviennent du .deploy du submodule Z3.Linq (serie Z3).
#r "../SymbolicAI/SMT/Z3.Linq/.deploy/Microsoft.Z3.dll"
using Microsoft.Z3;
using System.IO;
using System.Runtime.InteropServices;

// Z3 est une librairie native (libz3.dll) ; on la resout a cote du Microsoft.Z3.dll charge (meme dossier .deploy).
NativeLibrary.SetDllImportResolver(typeof(Context).Assembly, (name, assembly, path) => {
    if (name == "libz3") {
        string asmDir = Path.GetDirectoryName(typeof(Context).Assembly.Location);
        string[] cands = {
            Path.Combine(asmDir ?? ".", "libz3.dll"),
            Path.GetFullPath("../SymbolicAI/SMT/Z3.Linq/.deploy/libz3.dll"),
            "../SymbolicAI/SMT/Z3.Linq/.deploy/libz3.dll"
        };
        foreach (var c in cands) if (File.Exists(c) && NativeLibrary.TryLoad(c, out IntPtr h)) return h;
    }
    return IntPtr.Zero;
});
var ctx = new Context();
Console.WriteLine($"Z3 charge (native libz3 resolue, .deploy submodule Z3.Linq). Version {Microsoft.Z3.Version.FullVersion}.");


Z3 charge (native libz3 resolue, .deploy submodule Z3.Linq). Version Z3 4.12.2.0.


In [7]:
using Microsoft.Z3;
using System.Text;

// Resolution du Sudoku par Z3 : 81 entiers + Distinct sur lignes/colonnes/blocs.
// C'est le VRAI resolveur de production (le temoin = la grille solution).
static int[,] ResoudreSudokuZ3(Context ctx, int[,] puzzle, out long ms)
{
    var sw = System.Diagnostics.Stopwatch.StartNew();
    var cells = new IntExpr[9, 9];
    var solver = ctx.MkSolver();
    for (int r = 0; r < 9; r++)
        for (int c = 0; c < 9; c++)
        {
            cells[r, c] = (IntExpr)ctx.MkConst($"c_{r}_{c}", ctx.IntSort);
            solver.Assert(ctx.MkAnd(ctx.MkLe(ctx.MkInt(1), cells[r, c]),
                                    ctx.MkLe(cells[r, c], ctx.MkInt(9))));
            if (puzzle[r, c] != 0)
                solver.Assert(ctx.MkEq(cells[r, c], ctx.MkInt(puzzle[r, c])));
        }
    // Lignes et colonnes : tous distincts
    for (int i = 0; i < 9; i++)
    {
        var ligne = new IntExpr[9];
        var colonne = new IntExpr[9];
        for (int j = 0; j < 9; j++) { ligne[j] = cells[i, j]; colonne[j] = cells[j, i]; }
        solver.Assert(ctx.MkDistinct(ligne));
        solver.Assert(ctx.MkDistinct(colonne));
    }
    // Blocs 3x3 : tous distincts
    for (int br = 0; br < 3; br++)
        for (int bc = 0; bc < 3; bc++)
        {
            var bloc = new IntExpr[9];
            int k = 0;
            for (int r = 0; r < 3; r++)
                for (int c = 0; c < 3; c++)
                    bloc[k++] = cells[br * 3 + r, bc * 3 + c];
            solver.Assert(ctx.MkDistinct(bloc));
        }

    var res = new int[9, 9];
    if (solver.Check() == Status.SATISFIABLE)
    {
        var m = solver.Model;
        for (int r = 0; r < 9; r++)
            for (int c = 0; c < 9; c++)
                res[r, c] = ((IntNum)m.Eval(cells[r, c], true)).Int;
    }
    sw.Stop();
    ms = sw.ElapsedMilliseconds;
    return res;
}

// Rendu de la grille en UNE seule chaine (lignes uniformes, separateurs alignes) pour
// eviter les sauts de ligne fragmentes qui s'affichent mal sous Jupyter / GitHub.
static string RendreGrille(int[,] g)
{
    var sb = new StringBuilder();
    string sep = "------+-------+------";
    for (int r = 0; r < 9; r++)
    {
        for (int c = 0; c < 9; c++)
        {
            sb.Append(g[r, c]);
            if (c == 2 || c == 5) sb.Append(" | ");
            else if (c < 8)       sb.Append(' ');
        }
        sb.Append('\n');
        if (r == 2 || r == 5) sb.Append(sep).Append('\n');
    }
    return sb.ToString();
}

// Puzzle classique (exemple Wikipedia). 0 = case vide.
int[,] puzzle = {
    {5,3,0, 0,7,0, 0,0,0},
    {6,0,0, 1,9,5, 0,0,0},
    {0,9,8, 0,0,0, 0,6,0},
    {8,0,0, 0,6,0, 0,0,3},
    {4,0,0, 8,0,3, 0,0,1},
    {7,0,0, 0,2,0, 0,0,6},
    {0,6,0, 0,0,0, 2,8,0},
    {0,0,0, 4,1,9, 0,0,5},
    {0,0,0, 0,8,0, 0,7,9}
};

long msSolve;
var solution = ResoudreSudokuZ3(ctx, puzzle, out msSolve);
Console.WriteLine($"Z3 a produit un temoin (grille solution) en {msSolve} ms :\n");
Console.Write(RendreGrille(solution));

Z3 a produit un temoin (grille solution) en 50 ms :



5 3 4 | 6 7 8 | 9 1 2
6 7 2 | 1 9 5 | 3 4 8
1 9 8 | 3 4 2 | 5 6 7
------+-------+------
8 5 9 | 7 6 1 | 4 2 3
4 2 6 | 8 5 3 | 7 9 1
7 1 3 | 9 2 4 | 8 5 6
------+-------+------
9 6 1 | 5 3 7 | 2 8 4
2 8 7 | 4 1 9 | 6 3 5
3 4 5 | 2 8 6 | 1 7 9


### Interpretation : Z3 = production du temoin (le travail dur)

Z3 a **produit** la grille solution - le *temoin* - en quelques dizaines de millisecondes. C'est le barreau 2 de 2020, mais **sans le cap de 21 caracteres** : on appelle Z3 directement, pas via le chemin SFAz3 qui etait tronque. C'est ce resolveur qui entre dans le benchmark Sudoku multi-paradigmes (aux cotes d'Infer.NET, du PSO, du reseau de neurones) : il **resout reellement** le dataset.

C'est aussi le "cop-out" de l'ancienne version de ce notebook qu'on tue ici : non, "utiliser Z3 directement" n'est pas une degression honteuse par rapport aux automates - c'est le **complement indispensable** du verificateur RE#. *Solving* (Z3) et *matching* (RE#) sont les deux faces de la serie.

## 6b. Les deux murs tombent - mais la difficulte migre

Le barreau 2 (2020) butait sur **deux murs**, tous deux lies a la materialisation d'un automate : l'explosion du produit de DFA (mur 1) ET le cap du temoin a ~21 caracteres (mur 2, [issue #6](https://github.com/AutomataDotNet/Automata/issues/6)). Le **fork Automata** (modernisation net8.0, MyIntelligenceAgency) change de moteur : son `RegexToSMTConverter` compile la syntaxe de surface `&` / `~` vers la **theorie des chaines SMT-LIB 2.6** (`re.inter`, `re.comp`, `re.range`, `re.loop`, `str.to_re`) - le dialecte que Z3 consomme **directement**.

Consequence : Z3 raisonne symboliquement sur les chaines, **sans jamais construire le produit d'automates**. Le mur 1 (explosion DFA) n'est donc pas "leve" - il n'est *plus rencontre*. Et le mur 2 (cap) disparait : un temoin de 30 caracteres pour `[a-z]{30}` sort sans probleme.

Mais resoudre ne devient pas gratuit : la difficulte **migre** dans le solveur de chaines de Z3, dont le cout suit la **forme** de la contrainte.

| Echelle | Contrainte | Temoin via `&`/`~` -> theorie des chaines Z3 |
|---------|-----------|----------------------------------------------|
| **`A & ~B` general** (mot de passe, entree de test) | intersection + complement, 1D | **rapide** (~100 ms, cf [notebook 06](../SymbolicAI/SMT/Z3/06_Witness_Generation_Automata.ipynb)) |
| **1 ligne Sudoku** (9 cases) | intersection 10-way, 1D | tractable mais **lent** (~10 s, mesure ci-dessous) |
| **Grille 81 cases** | tous-distincts **2D** (lignes inter colonnes inter blocs) | **`unknown`** (mauvaise forme - cf section 8) |

La grille 81 n'est pas bloquee par un mur d'automate : elle est une contrainte **2D**, mauvaise forme pour la theorie des chaines 1D. Son bon encodage reste `Distinct` sur 81 entiers (section 6, ~47 ms) - une question de **representation**, pas de mur.

In [8]:
// Le fork Automata compile la MEME intersection 10-way que RE# verifie (section 5) vers la
// THEORIE DES CHAINES de Z3 (re.inter / re.comp / re.range), PAS vers un produit d'automates.
// On boucle la chaine SMT -> Z3 -> temoin, puis on CROISE les moteurs : le temoin doit etre
// VALIDE par le verificateur RE# `ligneValide` (section 5).
#r "../SymbolicAI/SMT/Automata/.deploy/System.CodeDom.dll"
#r "../SymbolicAI/SMT/Automata/.deploy/Microsoft.Automata.dll"
using System;
using System.Linq;
using System.Diagnostics;
using Microsoft.Automata;
using Microsoft.Z3;

var conv = new RegexToSMTConverter(BitWidth.BV7);
string ligneSurface = "^[1-9]{9}$&.*1.*&.*2.*&.*3.*&.*4.*&.*5.*&.*6.*&.*7.*&.*8.*&.*9.*";
string smtLigne = conv.ConvertRegex(ligneSurface);   // dialecte SMT-LIB 2.6 (theorie des chaines)
Console.WriteLine($"Le fork emet du SMT-LIB 2.6 ({smtLigne.Length} caracteres). Tete :");
Console.WriteLine("  " + smtLigne.Substring(0, Math.Min(150, smtLigne.Length)) + " ...");
Console.WriteLine();

// On resout ce terme regex via Z3 (ctx defini section 6) : (str.in_re w <terme>).
string script = "(declare-const w String)\n(assert (str.in_re w " + smtLigne + "))\n(check-sat)";
var sw = Stopwatch.StartNew();
var asserts = ctx.ParseSMTLIB2String(script);
var solver = ctx.MkSolver();
solver.Assert(asserts);
var status = solver.Check();
string temoinLigne = null;
if (status == Status.SATISFIABLE)
{
    var w = (SeqExpr)ctx.MkConst("w", ctx.StringSort);
    temoinLigne = solver.Model.Eval(w, true).ToString().Trim('"');
}
sw.Stop();

bool permutation = temoinLigne != null && temoinLigne.Length == 9 &&
                   Enumerable.Range(1, 9).All(d => temoinLigne.Contains((char)('0' + d)));
bool valideParREsharp = temoinLigne != null && ligneValide.Matches(temoinLigne).Length > 0;

Console.WriteLine($"Temoin Z3 (1 ligne, theorie des chaines) : {temoinLigne}   [{status}]");
Console.WriteLine($"  permutation de 1..9     : {permutation}");
Console.WriteLine($"  valide selon RE#        : {valideParREsharp}  (validation croisee inter-moteurs)");
Console.WriteLine($"  temps generation temoin : {sw.Elapsed.TotalMilliseconds:F0} ms");
Console.WriteLine();
Console.WriteLine("Mur 2 (cap ~21 char) : disparu - temoin complet de 9 caracteres.");
Console.WriteLine("Mur 1 (explosion DFA) : non rencontre - aucun produit d'automates n'est construit.");
Console.WriteLine("Mais la difficulte a MIGRE dans le solveur de chaines : de l'ordre de la dizaine");
Console.WriteLine("de secondes sur cette intersection 10-way (vs ~100 ms pour un 'A & ~B' general, NB06).");

Le fork emet du SMT-LIB 2.6 (1765 caracteres). Tete :


  (re.inter (re.++ (str.to_re "")(re.++ ((_ re.loop 9 9) (re.range "1" "9")) (str.to_re "")))(re.inter (re.++ (re.* (re.union (re.range "\u{0}" "\u{9}") ...


Temoin Z3 (1 ligne, theorie des chaines) : 129745836   [SATISFIABLE]


  permutation de 1..9     : True


  valide selon RE#        : True  (validation croisee inter-moteurs)


  temps generation temoin : 15072 ms


Mur 2 (cap ~21 char) : disparu - temoin complet de 9 caracteres.


Mur 1 (explosion DFA) : non rencontre - aucun produit d'automates n'est construit.


Mais la difficulte a MIGRE dans le solveur de chaines : de l'ordre de la dizaine


de secondes sur cette intersection 10-way (vs ~100 ms pour un 'A & ~B' general, NB06).


In [9]:
int groupes = 9 + 9 + 9;                        // 27
int sousContraintesParGroupe = 1 + 9;           // 1 longueur + 9 presences de chiffre
int total = groupes * sousContraintesParGroupe; // 270

Console.WriteLine($"Grille 81 cases = {groupes} groupes x {sousContraintesParGroupe} = {total} sous-contraintes.");
Console.WriteLine();
Console.WriteLine("Pourquoi la grille 81 ne passe pas par la theorie des chaines :");
Console.WriteLine("  - une SEULE ligne (intersection 10-way) demande deja ~10 s (mesure ci-dessus) ;");
Console.WriteLine("  - lignes, colonnes et blocs se CHEVAUCHENT (chaque case appartient a 3 groupes) :");
Console.WriteLine("    ce n'est pas une intersection 1D mais une contrainte 2D sur 81 caracteres ;");
Console.WriteLine("  - encodee en chaines, Z3 repond 'unknown' (> 60 s, cf section 8) : mauvaise FORME.");
Console.WriteLine();
Console.WriteLine("Verdict : changer de moteur fait tomber les deux murs de 2020 (cap + explosion DFA),");
Console.WriteLine("mais la difficulte MIGRE dans le solveur de chaines. Pour le Sudoku, le bon encodage");
Console.WriteLine("n'est pas un regex : c'est Distinct sur 81 entiers (section 6, ~47 ms).");
Console.WriteLine("Le vrai payoff de la generation de temoin 'A & ~B' est GENERAL");
Console.WriteLine("(mots de passe, entrees de test) - cf notebook 06 de la serie Z3.");

Grille 81 cases = 27 groupes x 10 = 270 sous-contraintes.


Pourquoi la grille 81 ne passe pas par la theorie des chaines :


  - une SEULE ligne (intersection 10-way) demande deja ~10 s (mesure ci-dessus) ;


  - lignes, colonnes et blocs se CHEVAUCHENT (chaque case appartient a 3 groupes) :


    ce n'est pas une intersection 1D mais une contrainte 2D sur 81 caracteres ;


...

Verdict : changer de moteur fait tomber les deux murs de 2020 (cap + explosion DFA),


mais la difficulte MIGRE dans le solveur de chaines. Pour le Sudoku, le bon encodage


n'est pas un regex : c'est Distinct sur 81 entiers (section 6, ~47 ms).


Le vrai payoff de la generation de temoin 'A & ~B' est GENERAL


(mots de passe, entrees de test) - cf notebook 06 de la serie Z3.


### Interpretation : les deux murs tombent, la difficulte se deplace

La chaine moderne **debride le temoin** : une ligne Sudoku (intersection 10-way) produit desormais un temoin reel, **valide par RE#** (validation croisee inter-moteurs ci-dessus) - chose impossible en 2020 sous le cap des 21 caracteres. Et aucun produit d'automates n'est construit, donc le "trop d'etats" de 2020 ne se produit pas. Les deux murs sont derriere nous.

Mais le gain a une **contrepartie honnete** : le cout passe dans le solveur de chaines de Z3. Rapide pour un `A & ~B` general (~100 ms), il devient lent sur l'intersection d'une ligne (~10 s) et **abandonne** (`unknown`) sur la grille 81 - non par mur, mais parce qu'un tous-distincts **2D** est la mauvaise forme pour la theorie des chaines **1D**. **Z3 reste le resolveur de production** via `Distinct` sur 81 entiers (~47 ms).

| Approche | Reconnaissance | Production de temoin | Echelle Sudoku 81 |
|----------|----------------|----------------------|-------------------|
| RE# (2025) | temps lineaire | aucun temoin | verifie seulement |
| `&`/`~` -> theorie des chaines Z3 | (oui) | **non cape, sans produit d'automates** | `unknown` (forme 2D) |
| Z3 `Distinct` (entiers) | - | temoin complet | **~47 ms (production)** |

> **Murs tombes, forme decisive.** Le payoff *general* de la generation de temoin `A & ~B` est demontre dans le notebook **[06 - Generer un temoin depuis A & ~B](../SymbolicAI/SMT/Z3/06_Witness_Generation_Automata.ipynb)** de la serie Z3 (mots de passe conformes, entrees de test, identifiants non utilises). Le Sudoku 81, lui, appartient a `Distinct` sur des entiers : ce n'est pas une affaire de mur, mais de **representation**.

## 7. L'ironie pedagogique - RE# valide ce qu'il ne peut produire

Nous avons maintenant les deux outils en main :

- **Z3** (section 6) a *produit* la grille solution (le temoin) ;
- **RE#** (section 5) sait *valider* une ligne.

Faisons-les se rencontrer : extrayons chaque ligne de la **solution produite par Z3**, et **verifions-la avec RE#**. Le verificateur elegant certifie, en temps lineaire, la solution qu'il est lui-meme **incapable de produire**.

In [10]:
using System.Diagnostics;
// Ironie : RE# valide CHAQUE ligne de la solution produite par Z3.
// Le verificateur elegant certifie - en temps lineaire - ce qu'il ne sait pas produire.
var lignesSolution = new string[9];
for (int r = 0; r < 9; r++) {
    char[] chars = new char[9];
    for (int c = 0; c < 9; c++) chars[c] = (char)('0' + solution[r, c]);
    lignesSolution[r] = new string(chars);
}
var sw = Stopwatch.StartNew();
int valides = 0;
foreach (var ligne in lignesSolution) {
    if (ligneValide.Matches(ligne).Length > 0) valides++;
}
sw.Stop();
Console.WriteLine($"RE# a verifie les 9 lignes de la solution Z3 en {sw.ElapsedTicks} ticks ({sw.ElapsedMilliseconds} ms).");
Console.WriteLine($"Lignes valides : {valides}/9");
Console.WriteLine();
Console.WriteLine("L'ironie : RE# certifie, plus vite que Z3 n'a resolu, une grille");
Console.WriteLine("qu'il ne SAURAIT PAS produire lui-meme (il est recognition-only).");
Console.WriteLine();
foreach (var ligne in lignesSolution) Console.WriteLine($"  {ligne} -> {(ligneValide.Matches(ligne).Length > 0 ? "VALIDE" : "invalide")}");

RE# a verifie les 9 lignes de la solution Z3 en 238767 ticks (23 ms).


Lignes valides : 9/9


L'ironie : RE# certifie, plus vite que Z3 n'a resolu, une grille


qu'il ne SAURAIT PAS produire lui-meme (il est recognition-only).


  534678912 -> VALIDE


...

  426853791 -> VALIDE


  713924856 -> VALIDE


  961537284 -> VALIDE


  287419635 -> VALIDE


  345286179 -> VALIDE


### Interpretation : le prestige va au verificateur, le travail au resolveur

Double retournement :

1. Le **reconnaisseur le plus elegant** (RE#, temps lineaire) ne sait **pas fabriquer** la solution qu'il certifie.
2. Il la certifie en outre **plus vite** que la toolchain (Automata / intersection DFA) qui etait censee etre *le* reconnaisseur en 2020 - celle-la meme qui explosait.

Le Sudoku donne a voir que **matching** (RE#) et **solving** (Z3) sont deux metiers. Le prestige du "rapide et elegant" va au verificateur ; le travail dur - la **production du temoin** - reste irremplacablement du cote de Z3. C'est la these de cette serie, rendue concrete.

## Exercice 2 : etendre l'encodage Z3 (Sudoku diagonal)

# Objectif
Le Sudoku-X ajoute deux contraintes : les deux **diagonales** principales doivent aussi contenir 1..9 (tous distincts). Etendez la fonction `ResoudreSudokuZ3` pour ajouter ces deux contraintes.

# Indice
Recuperez les 9 `IntExpr` de la diagonale principale (`cells[i,i]`) et de l'anti-diagonale (`cells[i, 8-i]`), et ajoutez deux `ctx.MkDistinct(...)` supplementaires avant `s.Check()`.

# Etape 1
Ecrivez une variante `ResoudreSudokuXZ3` dans la cellule suivante.

In [11]:
// EXERCICE : ResoudreSudokuXZ3 (variante avec contraintes de diagonales)
// TODO etudiant : reprendre le schema de ResoudreSudokuZ3 et ajouter deux MkDistinct
// sur la diagonale principale (cells[i,i]) et l'anti-diagonale (cells[i, 8-i]).
public int[,] ResoudreSudokuXZ3(Context ctx, int[,] puzzle)
{
    return puzzle;  // TODO etudiant : remplacer par la resolution avec contraintes diagonales
}
Console.WriteLine("Exercice a completer : implementer les deux contraintes de diagonale.");

Exercice a completer : implementer les deux contraintes de diagonale.


## 8. Resoudre par toutes les voies - le banc d'essai

Jusqu'ici chaque barreau a ete illustre isolement. Cette section les **execute cote a cote** sur les memes grilles : la canonique (30 indices), la grille vide (0 indice) et une grille B a 26 indices. L'objectif n'est PAS de trouver l'optimum - pour ca il y a Dancing Links (DLX), et on remballe. L'objectif est que **la discussion soit fertile** : on tente toutes les resolutions, les bonnes comme les moins bonnes, et un echec honnete (timeout, faux negatif) en dit autant qu'une reussite.

### Conway : deux artefacts a ne pas confondre

Le "Sudoku en un seul regex" de la lignee Conway existe en **deux versions distinctes** qu'on melange souvent a tort :

| Variante | Auteur | Mecanisme | Tourne sur |
|----------|--------|-----------|------------|
| **Monstre recursif** | Damian Conway (2005), Davidebyzero | recursion PCRE `(?R)` / `(?0)` - **non regulier** | PCRE, module Python `regex` ; **PAS** `System.Text.RegularExpressions` |
| **Variante deroulee** | Aron Griffis (2007), domaine public | recursion **deroulee** en 81 blocs explicites : backrefs `\1`..`\81` + lookaheads, **aucun** `(?R)` | tout moteur a backtracking : stdlib Python `re` **et** .NET |

Le monstre recursif est l'**anti-modele** du barreau 1 : elegant, illisible, et hors de portee du moteur .NET, qui ne sait pas faire de recursion de motif. (Le mecanisme `(?R)` est verifiable en une ligne dans le module Python `regex` : `\((?:[^()]|(?R))*\)` reconnait les parentheses bien balancees - un langage non regulier - mais ne compile meme pas en .NET.) C'est donc la **variante deroulee de Griffis** qu'on execute ci-dessous en .NET : elle, au moins, tourne. La cellule genere les 81 blocs, puis resout par UNE substitution `Regex.Replace` - c'est le moteur de backtracking qui fait toute la recherche.

In [12]:
using System;
using System.Linq;
using System.Text;
using System.Diagnostics;
using System.Text.RegularExpressions;

// --- Generateur de la variante DEROULEE (Aron Griffis 2007, lignee Conway, domaine public) ---
// 81 blocs explicites, forward-check par lookahead, AUCUNE recursion (?R) : tourne en .NET.
int[] BlocDe(int i) { int x = i % 9, y = i / 9, b = (y / 3) * 27 + (x / 3) * 3;
    return new[] { b, b+1, b+2, b+9, b+10, b+11, b+18, b+19, b+20 }; }

string BuildSudokuRegex() {
    var re = new StringBuilder(); re.Append(@"\A"); re.Append("\n\n");
    for (int i = 0; i < 81; i++) {
        re.Append(@"\d*");
        var col = Enumerable.Range(0, i/9).Select(y => y*9 + (i%9));
        var row = Enumerable.Range(0, i%9).Select(x => (i/9)*9 + x);
        var blc = BlocDe(i).Where(c => c < i);
        var deja = col.Concat(row).Concat(blc).Distinct().OrderBy(z => z).ToList();
        if (deja.Count > 0) { re.Append(@"(?!" + string.Join("|", deja.Select(p => @"\" + (p+1))) + ")"); re.Append("\n"); }
        re.Append(@"(\d)"); re.Append("\n");                                                        // la case : on CAPTURE un chiffre
        re.Append(@"(?!(?:.*\n)+(?:.{10}){" + (i%9) + @"}\" + (i+1) + @"\b)"); re.Append("\n");      // pas ce chiffre dans la colonne en dessous
        re.Append(@"(?!\d*\ (?:.{10})*?\" + (i+1) + @"\b)"); re.Append("\n");                        // ni ailleurs apres, sur n'importe quelle ligne
        if (i%3 < 2) { re.Append(@"(?!\d*\ (?:.{10}){0," + (1 - i%3) + @"}\" + (i+1) + @"\b)"); re.Append("\n"); }
        if ((i/9)%3 < 2) { re.Append(@"(?!(?:.*\n){1," + (2 - (i/9)%3) + @"}(?:.{30}){" + ((i%9)/3) + @"}(?:.{10}){0,2}\" + (i+1) + @"\b)"); re.Append("\n"); }
        re.Append(@"\d*\s+"); re.Append("\n\n");
    }
    re.Append(@"\Z"); re.Append("\n"); return re.ToString();
}

// Resout par UNE substitution regex : c'est le moteur de backtracking qui fait la recherche.
(string grille, double ms) ResoudreParRegex(string puzzle, Regex rx) {
    string spread = Regex.Replace(puzzle, "[1-9]", "$0        ");  // chiffre + 8 espaces -> case large de 10
    string menu   = Regex.Replace(spread, "0", "123456789");       // case vide -> menu de candidats 1..9
    var repl = new StringBuilder();
    for (int r = 0; r < 9; r++) { for (int c = 0; c < 9; c++) { if (c > 0) repl.Append(' '); repl.Append("${" + (r*9+c+1) + "}"); } if (r < 8) repl.Append('\n'); }
    var sw = Stopwatch.StartNew();
    try {
        string outp = rx.Replace(menu, repl.ToString()); sw.Stop();
        bool solved = !outp.Replace(" ", "").Replace("\n", "").Contains("123456789");
        return (solved ? outp.Trim() : null, sw.Elapsed.TotalMilliseconds);
    } catch (RegexMatchTimeoutException) { sw.Stop(); return ("__TIMEOUT__", sw.Elapsed.TotalMilliseconds); }
}

string canon   = "5 3 0 0 7 0 0 0 0\n6 0 0 1 9 5 0 0 0\n0 9 8 0 0 0 0 6 0\n8 0 0 0 6 0 0 0 3\n4 0 0 8 0 3 0 0 1\n7 0 0 0 2 0 0 0 6\n0 6 0 0 0 0 2 8 0\n0 0 0 4 1 9 0 0 5\n0 0 0 0 8 0 0 7 9";
string vide    = string.Join("\n", Enumerable.Repeat("0 0 0 0 0 0 0 0 0", 9));
string grilleB = "0 4 5 0 1 0 0 0 0\n0 0 0 7 5 0 0 0 8\n0 0 7 0 8 0 0 1 0\n0 0 0 0 0 7 0 0 6\n6 8 0 0 0 0 0 7 2\n1 0 0 3 0 0 0 0 0\n0 6 0 0 7 0 8 0 0\n7 0 0 0 9 1 0 0 0\n0 0 0 0 3 0 4 5 0";

string pat = BuildSudokuRegex();

// Persiste l'artefact REEL : c'est LA "vraie chose" affichee tronquee en section 2 (cell 3).
// Le fichier committe en est byte-identique (meme generateur), la regen est idempotente.
try {
    var dir = new[] { "assets", "MyIA.AI.Notebooks/Sudoku/assets" }.FirstOrDefault(System.IO.Directory.Exists) ?? "assets";
    System.IO.Directory.CreateDirectory(dir);
    System.IO.File.WriteAllText(System.IO.Path.Combine(dir, "sudoku-unrolled.regex.txt"), pat);
    Console.WriteLine($"Artefact sauvegarde : assets/sudoku-unrolled.regex.txt ({pat.Length} caracteres).");
} catch { Console.WriteLine("(assets/ en lecture seule : l'artefact committe fait foi)"); }

var rx = new Regex(pat, RegexOptions.IgnorePatternWhitespace, TimeSpan.FromSeconds(10));  // cap match 10 s = aveu d'echec honnete
Console.WriteLine($"Regex genere : {pat.Length} caracteres, 81 blocs, 0 recursion (?R).");
Console.WriteLine("Moteur : System.Text.RegularExpressions (.NET) ; cap match = 10 s.\n");

foreach (var (nom, puz) in new[] { ("canonique (30 indices)", canon), ("grille vide (0 indice)", vide), ("grille B (26 indices)", grilleB) }) {
    var (grille, ms) = ResoudreParRegex(puz, rx);
    string verdict = grille == "__TIMEOUT__" ? $"TIMEOUT (> {ms/1000:F0} s)"
                   : grille == null          ? $"PAS RESOLU - faux negatif en {ms:F0} ms"
                   :                            $"RESOLU en {ms:F1} ms";
    Console.WriteLine($"--- {nom} : {verdict} ---");
    Console.WriteLine(grille != null && grille != "__TIMEOUT__" ? grille + "\n" : "");
}

Artefact sauvegarde : assets/sudoku-unrolled.regex.txt (13515 caracteres).


Regex genere : 13515 caracteres, 81 blocs, 0 recursion (?R).


Moteur : System.Text.RegularExpressions (.NET) ; cap match = 10 s.



--- canonique (30 indices) : RESOLU en 4,8 ms ---


5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9



--- grille vide (0 indice) : TIMEOUT (> 10 s) ---


--- grille B (26 indices) : PAS RESOLU - faux negatif en 860 ms ---


### Le meme regex, deux moteurs : la portabilite est une illusion

Le motif genere ci-dessus fait 13515 caracteres et ne contient **aucune** recursion : il devrait donc se comporter pareil partout. Il n'en est rien. Le **meme** motif, octet pour octet, applique en .NET (`System.Text.RegularExpressions`, cellule ci-dessus) et en Python (`re`, lignee PCRE, mesure reproductible hors notebook) **diverge** :

| Grille | .NET `System.Text.RegularExpressions` | Python `re` |
|--------|---------------------------------------|-------------|
| canonique (30 indices) | RESOLU en **2,8 ms** | RESOLU en 14 ms |
| grille vide (0 indice) | **TIMEOUT (> 10 s)** | RESOLU en 11 ms |
| grille B (26 indices) | **PAS RESOLU - faux negatif (1666 ms)** | RESOLU en 607 ms |

Sur la grille canonique, .NET est meme **plus rapide**. Mais sur la grille vide il **boucle** jusqu'au cap de 10 s, et sur la grille B il rend un **faux negatif** : il declare "pas de solution" la ou il en existe une que Python trouve. Les lookaheads d'optimisation que Griffis a regles pour PCRE - `\b`, `*?` paresseux, semantique du `.` face au saut de ligne - ne portent pas la **meme semantique** chez le backtracker .NET.

> **Lecon.** Un regex de **resolution** n'est pas portable : il est accorde a un moteur precis. Le "reconnaitre != resoudre" du reste du notebook se double ici d'un "le meme motif ne calcule pas la meme chose selon le moteur".

In [13]:
using System;
using System.Linq;
using System.Diagnostics;

// Le contre-point : backtracking RECURSIF naif. La pile d'appels EST la pile de contexte
// (legere, zero allocation par essai). C'est l'outil qui epouse le substrat - et il est robuste.
bool Ok(int[] g, int i, int d) {
    int r = i/9, c = i%9, br = 3*(r/3), bc = 3*(c/3);
    for (int k = 0; k < 9; k++) if (g[r*9+k] == d || g[k*9+c] == d) return false;
    for (int dr = 0; dr < 3; dr++) for (int dc = 0; dc < 3; dc++) if (g[(br+dr)*9+bc+dc] == d) return false;
    return true;
}
bool Resoudre(int[] g) {
    int i = Array.IndexOf(g, 0); if (i < 0) return true;          // plus de case vide -> resolu
    for (int d = 1; d <= 9; d++) if (Ok(g, i, d)) { g[i] = d; if (Resoudre(g)) return true; g[i] = 0; }
    return false;                                                  // aucun chiffre ne passe -> on remonte (backtrack)
}
void Banc(string nom, string s81) {
    var g = s81.Where(ch => ch != ' ' && ch != '\n').Select(ch => (ch == '.' || ch == '0') ? 0 : ch - '0').ToArray();
    var sw = Stopwatch.StartNew(); bool ok = Resoudre(g); sw.Stop();
    Console.WriteLine($"{nom,-22} : {(ok ? "RESOLU" : "ECHEC")} en {sw.Elapsed.TotalMilliseconds:F1} ms");
}
Banc("canonique (30)", "53..7....6..195....98....6.8...6...34..8.3..17...2...6.6....28....419..5....8..79");
Banc("grille vide (0)", new string('.', 81));
Banc("grille B (26)",   "045010000000750008007080010000007006680000072100300000060070800700091000000030450");

canonique (30)         : RESOLU en 1,1 ms


grille vide (0)        : RESOLU en 0,1 ms


grille B (26)          : RESOLU en 2,9 ms


### Le banc d'essai complet - et pourquoi les echecs sont fertiles

Toutes les voies, mesurees (kernel `.net-csharp` pour les lignes C#/.NET ; z3-py et Python `re` reproductibles hors notebook) :

| Voie | Substrat | canon (30) | vide (0) | grille B (26) |
|------|----------|-----------:|---------:|--------------:|
| Naif recursif | **C#** | 1,4 ms | **0,1 ms** | 3,6 ms |
| Naif recursif | Python | 16,3 ms | 1,7 ms | 43,8 ms |
| Naif recursif | NumPy | 106 ms | - | 311 ms |
| Regex deroule | **.NET** | 2,8 ms | **TIMEOUT > 10 s** | **faux negatif** |
| meme regex | Python `re` | 14 ms | 11 ms | 607 ms |
| P1 Z3 `Distinct` | z3-py | 23,3 ms | **25,2 s** | 74,4 ms |
| P3 Z3 chaines (`str.in_re`) | z3-py | **unknown > 60 s** | - | - |
| P2 DFA -> SMT | .NET (fork) | OOM a 81 (mur 1, cf. section 6b) | - | - |
| RE# | .NET | reconnaissance seule, **aucun temoin** | - | - |
| Monstre Conway `(?R)` | PCRE / Python `regex` | non regulier, **absent de .NET** | - | - |
| Dancing Links (DLX) | - | l'optimum (ordre de la microseconde) - **hors perimetre** | - | - |

*Lecture : une valeur en ms = resolu ; TIMEOUT / faux negatif / unknown / OOM = echec honnete. Pour Z3 `Distinct`, la ligne ci-dessus est mesuree en z3-py sur les trois grilles ; le meme encodage tourne en C# en direct a la section 6 (de l'ordre de 47 ms sur la canonique, binding Z3 .NET).*

**Quatre lecons fertiles** - c'est la discussion, pas le chrono, qui est le livrable :

1. **Le substrat compte autant que l'algorithme.** Le *meme* backtracking naif fait 0,1 ms (C#), 1,7 ms (Python) et 106 ms (NumPy) sur la grille vide. La pile d'appels C# *est* une pile de contexte legere (zero allocation par essai) ; NumPy paie un surcout par-element a chaque acces scalaire - la vectorisation ne sert a rien sur une recherche sequentielle, elle nuit. L'outil doit epouser le probleme.

2. **L'anti-modele "gagne" puis s'effondre.** Le regex deroule est competitif sur la canonique (2,8 ms, du meme ordre que le naif C#, loin devant Z3) - puis il *timeout* sur la grille vide et rend un *faux negatif* sur la grille B. Rapide la ou c'est facile, faux la ou c'est dur : exactement le profil qu'on ne veut pas d'un resolveur.

3. **Le naif recursif est le vainqueur discret.** 1,4 / 0,1 / 3,6 ms : rapide *et* robuste sur les trois grilles, sans aucune machinerie symbolique. Le bon vieux backtracking, bien empile en recursif, n'a pas du tout des performances mediocres - il bat meme le regex sur la canonique. Les metaheuristiques (recuit, genetique) qui mettent plus d'une minute sur ce probleme sont, ici, un contresens d'outil.

4. **Les voies "elegantes symboliques" sont precisement celles qui explosent.** P2 (DFA -> SMT) explose en etats (mur 1, section 6b), P3 (regex -> theorie des chaines Z3) ne termine pas (`unknown` apres 60 s), le monstre Conway recursif n'existe pas en .NET. La seule voie symbolique tractable - Z3 `Distinct` - est celle qui **abandonne la representation regex** pour revenir a 81 entiers et une seule contrainte `Distinct`. Le pouvoir vient du bon niveau d'abstraction, pas du plus spectaculaire.

> **Synthese du banc.** Reconnaitre (RE#) est lineaire et portable ; resoudre par regex est fragile et non portable ; resoudre proprement - Z3 `Distinct`, ou meme un simple backtracking C# - demande de **choisir la bonne abstraction**. Le DLX serait l'optimum, mais on ne cherchait pas l'optimum : on cherchait a comprendre *pourquoi* chaque voie reussit ou echoue. Voir les sections 6 et 6b (Z3 et le fork Automata) et 4-5 (RE# reconnaissance).

## 9. Synthese - reconnaitre != resoudre

Les deux murs de 2020 etaient des murs **de l'automate** ; ils tombent ensemble des qu'on change de moteur. Mais aucun outil ne couronne l'echelle : la difficulte se **deplace**, et le bon outil suit la **forme** du probleme.

| | **RESOUDRE** (produire la grille) | **VERIFIER** (valider une grille remplie) |
|---|---|---|
| Folklore PCRE (backtracking) | detour de backtracking, illisible | monstrueux |
| BREX/Rex + SFAz3 (2020) | **mure** (cap ~21 char, issue #6) | **explosion DFA** |
| `&`/`~` -> theorie des chaines Z3 (moderne) | temoin non-cape, sans produit d'automates ; rapide pour `A & ~B`, lent pour une ligne, `unknown` a 81 | (oui) |
| RE# (2025) | recognition-only, **aucun temoin** | **temps lineaire** |
| Z3 `Distinct` (81 entiers) | **resolveur reel du benchmark** (~47 ms) | - |

**Lecon** : decrire une contrainte par intersection reguliere (`Ligne & Colonne & Bloc`) est une representation elegante, mais elle ne dispense pas d'un resolveur adapte a la *forme* du probleme pour *produire* la solution. Le Sudoku - un tous-distincts **2D** - a besoin de `Distinct` sur des entiers (Z3) ; RE# en est le verificateur, pas le solveur. L'ironie (RE# valide plus vite le temoin qu'il ne peut produire) est le pont entre cette serie Sudoku et la serie Z3 (Epic #1206).

La **chaine moderne** (section 6b, et notebook [06](../SymbolicAI/SMT/Z3/06_Witness_Generation_Automata.ipynb) de la serie Z3) fait tomber **les deux murs** de 2020 : `&`/`~` compile vers la theorie des chaines de Z3, qui ne cape aucun temoin et ne construit aucun produit d'automates. Le revers honnete : sur la grille 81 (forme 2D) Z3 string-theory repond `unknown` - Z3 `Distinct` sur des entiers reste donc le resolveur de production.

Le **banc d'essai** (section 8 ci-dessus) confirme empiriquement chaque case de ce tableau : le regex deroule s'effondre (timeout, faux negatif), Z3 `Distinct` reste robuste, et le backtracking C# naif est le vainqueur discret.

> **Footnote d'homonymie** : **Damian** Conway (luminaire des regex Perl, dont releve le folklore du barreau 1) n'est pas **John** Conway (le Game of Life, hero de notre serie Lean #2162). Les deux Conway se croisent dans ce notebook.

## Exercice 3 : classifier recognition vs solving (conceptuel)

# Objectif
Pour chacune des taches ci-dessous, indiquez quel outil convient (RE# pour la reconnaissance / verification, Z3 pour la resolution / production de temoin) et pourquoi.

| Tache | Outil attendu | Pourquoi |
|-------|---------------|----------|
| (a) Verifier qu'une grille remplie respecte les regles | RE# | ... |
| (b) Resoudre un puzzle a partir de cases vides | Z3 | ... |
| (c) Verifier qu'une chaine est un nombre a 9 chiffres distincts | ... | ... |
| (d) Trouver une permutation de 1..9 maximisant un critere | ... | ... |

# Indice
Posez-vous : la tache demande-t-elle de *produire* une solution (solving -> Z3) ou seulement de *reconnaitre* si une entree donnee est valide (matching -> RE#) ?

In [14]:
// EXERCICE (conceptuel) : classifier recognition (RE#) vs solving (Z3).
// TODO etudiant : completer le tableau avec l'outil et la justification.
Console.WriteLine("Exercice a completer : pour chaque tache, indiquer RE# ou Z3 et pourquoi.");
Console.WriteLine();
Console.WriteLine("(a) Verifier qu'une grille remplie respecte les regles : RE#  -> ...");
Console.WriteLine("(b) Resoudre un puzzle a partir de cases vides       : Z3  -> ...");
Console.WriteLine("(c) Verifier qu'une chaine = 9 chiffres distincts    : ... -> ...");
Console.WriteLine("(d) Trouver une permutation de 1..9 maximisant un critere : ... -> ...");

Exercice a completer : pour chaque tache, indiquer RE# ou Z3 et pourquoi.


(a) Verifier qu'une grille remplie respecte les regles : RE#  -> ...


(b) Resoudre un puzzle a partir de cases vides       : Z3  -> ...


(c) Verifier qu'une chaine = 9 chiffres distincts    : ... -> ...


(d) Trouver une permutation de 1..9 maximisant un critere : ... -> ...


---

## 9. References & connexions

### Sources primaires
- **Folklore "Sudoku en un regex"** (tradition Perl) - [ikegami, PerlMonks 2005](https://www.perlmonks.org/?node_id=471168) (solveur via le moteur regex avec blocs de code embarques) ; module pur-regex [`Regexp::Sudoku` d'Abigail](https://metacpan.org/pod/Regexp::Sudoku). Le monstre PCRE (barreau 1), en formes recursive `(?R)` (hors .NET) et deroulee (tourne en .NET).
- **Forme deroulee (generateur)** - Aron Griffis, *Sudoku with a Perl regular expression* (2007), domaine public : les 81 blocs explicites (lookaheads + backreferences, sans `(?R)`) que nous **regenerons et executons** en section 8 (artefact `assets/sudoku-unrolled.regex.txt`).
- **Issue upstream** [AutomataDotNet/Automata#6](https://github.com/AutomataDotNet/Automata/issues/6) (creee 2020-12-07, toujours sans reponse) : cap du temoin a ~21 caracteres (barreau 2, mur 2).
- **Margus Veanes et al.**, *Derivative-based nonbacktracking real-world regex matching* - [MSR-TR-2020-25](https://www.microsoft.com/en-us/research/publication/derivative-based-nonbacktracking-real-world-regex-matching-with-backtracking-semantics/) (aout 2020). Le papier contemporain de la tentative 2020.
- **RE#** (engine F#/.NET, MIT) - [github.com/ieviev/resharp-dotnet](https://github.com/ieviev/resharp-dotnet), NuGet `Resharp`. `&`/`~` first-class, non-backtracking, temps lineaire (barreau 3).
- **Fork Automata** (net8.0, MyIntelligenceAgency) - [github.com/MyIntelligenceAgency/Automata](https://github.com/MyIntelligenceAgency/Automata). Modernise AutomataDotNet (gele ~2020) : son `RegexToSMTConverter` compile `&`/`~` de surface vers la theorie des chaines SMT-LIB 2.6 (`re.inter`/`re.comp`) que Z3 resout directement - cap du temoin (#6) leve, plus de produit d'automates. Consomme par le notebook 06 de la serie Z3.

### Connexions dans cette serie
- **[Sudoku-12 Z3 C#](Sudoku-12-Z3-Csharp.ipynb)** : le resolveur Z3 explore en profondeur (bit-vectors, optimisations). Ce notebook (13) presente le meme Z3 sous l'angle narratif "regex symbolique -> SMT -> temoin".
- **Epic #1206 (Z3.Linq DSL)** : un autre "geste" de l'auteur (2018), etendre un front-end declaratif pour laisser Z3 temoigner. La compilation regex -> SMT et le DSL Z3.Linq sont **le meme geste dans deux librairies**.
- **[Notebook 06 - Generer un temoin depuis A & ~B](../SymbolicAI/SMT/Z3/06_Witness_Generation_Automata.ipynb)** : le fork Automata leve le cap du temoin (#6) et generalise `A & ~B` (mots de passe, entrees de test) - le cas ou la theorie des chaines de Z3 est rapide. La section 6b ci-dessus mesure le spectre (rapide en general, lent sur une ligne, `unknown` a 81).
- **Epic #2162 (Game of Life, Lean)** : **John** Conway, a ne pas confondre avec **Damian** (regex).

### La preuve Lean derriere RE#
- **[ezhuchko/finiteness-derivatives](https://github.com/ezhuchko/finiteness-derivatives)** (Lean 4, sans Mathlib) : formalise la **finitude des derivees symboliques** = le theoreme de terminaison/decidabilite derriere la reconnaissance non-backtracking temps lineaire de RE#. Companion CPP 2024 (Zhuchko / Veanes / Ebner).

---

## A retenir

- **Reconnaitre != resoudre** : RE# verifie (temps lineaire), Z3 produit (le temoin). Le Sudoku a besoin des deux.
- Les deux murs de 2020 (explosion DFA, cap temoin ~21 char) etaient des murs **de l'automate** : ils tombent **ensemble** des qu'on change de moteur (`&`/`~` -> theorie des chaines Z3).
- Mais resoudre ne devient pas gratuit : la difficulte **migre** dans le solveur de chaines - rapide pour `A & ~B`, lente pour une ligne Sudoku, `unknown` pour la grille 81.
- Pour le Sudoku, le bon encodage n'est pas un regex : c'est `Distinct` sur 81 entiers (~47 ms). Question de **forme**, pas de mur.
- L'**ironie** : le verificateur le plus elegant (RE#) certifie, plus vite que Z3 n'a resolu, une grille qu'il ne peut pas produire.